# Цели:
С помощью function calling:
1. отправить сообщение в телеграм
2. получить информацию о событиях за какой-то день с помощью апи гугл календаря

## Функция для отправки сообщения

In [ ]:
pip install openai -q

In [2]:
import requests
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

In [3]:
load_dotenv()
tg_api_key=os.getenv("TELEGRAM_BOT_API_KEY")

In [4]:
def load_db():
    '''tg_users - словарь, вида:
    {	"username": 1234    }'''
    with open('tg_users.txt','r',encoding="utf-8") as file:
        return json.load(file)

def get_chat_id(username):
    db = load_db()
    return db.get(username)

In [5]:
def send_telegram_msg(username:str, content:str):
    chat_id = get_chat_id(username)
    send_message_url = f"https://api.telegram.org/bot{tg_api_key}/sendMessage"
    payload = {
        'chat_id':chat_id,
        'text':content
    }
    response = requests.post(send_message_url, data=payload)
    data = response.json()
    print(data)
    return {
        "recipient": data['result']['chat']['username'],
        'text': data['result']['text']
    }

## Функция для календаря

In [ ]:
!pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

In [8]:
import datetime
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [10]:
def show_google_calendar_events(start_date=None, end_date=None):
    """ basic usage of the Google Calendar API.
    Prints the start and name of the next 10 events on the user's calendar.
    """
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "credentials.json", SCOPES
            )
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("calendar", "v3", credentials=creds)

        # Call the Calendar API
        if start_date is None:
            time_min = datetime.datetime.now(tz=datetime.timezone.utc).isoformat()
        else:
            time_min = datetime.datetime.fromisoformat(start_date).replace(
                tzinfo=datetime.timezone.utc
            ).isoformat()

        if end_date is None:
            time_max = None
        else:
            time_max = (
                datetime.datetime.fromisoformat(end_date).replace(
                    tzinfo=datetime.timezone.utc
                ) + datetime.timedelta(days=1)
            ).isoformat()

        print("Getting the upcoming 10 events")
        events_result = (
            service.events()
            .list(
                calendarId="primary",
                timeMin=time_min,
                timeMax=time_max,
                maxResults=10,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        events = events_result.get("items", [])

        if not events:
            print("No upcoming events found.")
            return

        # Prints the start and name of the next 10 events
        for event in events:
            start = event["start"].get("dateTime", event["start"].get("date"))
            print(start, event["summary"])

    except HttpError as error:
        print(f"An error occurred: {error}")
    
    return {
        "events": [
            {
                "start": event["start"].get("dateTime", event["start"].get("date")),
                "summary": event["summary"],
            }
            for event in events
        ]
    }

In [ ]:
SCOPES = ["https://www.googleapis.com/auth/calendar.readonly"]

# start_date: строка 'YYYY-MM-DD' или None
# end_date:   строка 'YYYY-MM-DD' или None

# Подготовка схемы

Нужно сформировать схему для функции, понятную модели.
Схема описывает:
- название функции
- описание что делает функция
- описание параметров

In [20]:
functions = [
    {
        "type": "function",
        "function": {
            "name": "send_telegram_msg",
            "description": "Отправить указанное сообщение в телеграм указанному пользователю",
            "parameters": {
                "type": "object",
                "properties": {
                    "username": {
                        "type": "string",
                        "description": "Имя пользователя с помощью которого ищется chat_id",
                    },
                    "content": {
                        "type": "string", 
                        "description": "Содержание сообщения"
                    },
                },
                "required": ["username","content"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "show_google_calendar_events",
            "description": "Показать все события из гугл-календаря за определенный период",
            "parameters": {
                "type": "object",
                "properties": {
                    "start_date": {
                        "type": "string",
                        "description": """Начальная дата интервала для поиска событий в формате 'YYYY-MM-DD'. 
                                        Если значение None, используется текущая дата. 
                                        Если пользователь не указал год, используй текущий год.""",
                    },
                    "end_date": {
                        "type": "string", 
                        "description": """Конечная дата интервала для поиска событий в формате 'YYYY-MM-DD'. 
                                        Если значение None, используется текущая дата. 
                                        Если пользователь не указал год, используй текущий год."""
                    },
                },
                "required": [],
            },
        }
    }
]

In [37]:
openai_client = OpenAI(
    api_key=os.getenv('OPENROUTER7_API_KEY'),
    base_url="https://openrouter.ai/api/v1"
)

# content='Напиши пользователю <username> "Помидоры помидоры"'
content='Какие события запланированы в google calendar с 23 апреля по 25 апреля 2026 года?'
# content='Привет'

request = {
    'model':"nvidia/nemotron-3-super-120b-a12b:free",
    'tools':functions,
    'messages':[
    {
        "role": "system",
        "content": "Проанализируй сообщение пользователя и реши какие функции стоит вызвать."
    },
    {
        "role": "user",
        "content":content 
    }
    ] 
}

response = openai_client.chat.completions.create(**request)

In [ ]:
# делаем красивое ответное сообщение
response_message = json.loads(response.model_dump_json(indent=3))
response_message = response_message['choices'][0]['message']
print(response_message)

# Реальное выполнение функции 

In [39]:
messages = []

tool_calls = response_message.get("tool_calls") or []

for tool_call in tool_calls:
    call_id = tool_call.get("id")
    fn_res = None

    if fn_call := tool_call.get("function"):
        fn_name = fn_call.get("name")
        fn_args = json.loads(fn_call.get("arguments", "{}"))

        if fn_name == "send_telegram_msg":
            fn_res = send_telegram_msg(
                username=fn_args.get("username"),
                content=fn_args.get("content")
            )

        elif fn_name == "show_google_calendar_events":
            fn_res = show_google_calendar_events(
                start_date=fn_args.get("start_date"),
                end_date=fn_args.get("end_date")
            )

    if fn_res is not None and call_id is not None:
        messages.append({
            "role": "tool",
            "content": fn_res,
            "tool_call_id": call_id,
        })

Getting the upcoming 10 events
2026-04-24T10:15:00-04:00 сдать лабу


In [40]:
messages

[{'role': 'tool',
  'content': {'events': [{'start': '2026-04-24T10:15:00-04:00',
     'summary': 'сдать лабу'}]},
  'tool_call_id': 'chatcmpl-tool-9be53db482b951a6'}]

In [41]:
second_request = {
    'model':"nvidia/nemotron-3-super-120b-a12b:free",
    'tools':functions,
    'messages':[
    {
        "role": "assistant",
        "content": None, 
        "tool_calls": [{
                "id": tool_call["id"],
                "type": "function",
                "function": {
                    "name": fn_name,
                    "arguments": tool_call["function"]["arguments"]
                }
        }]
    },
    {
        "role": "tool",
        "content": json.dumps(fn_res),
        "tool_call_id": tool_call["id"]
    },
    {
        "role": "user",
        "content":content 
    }
    ] 
}

second_response = openai_client.chat.completions.create(**second_request)

In [42]:
print(second_response.choices[0].message.content)

В указанный период (с 23 по 25 апреля 2026 г.) в вашем Google Calendar запланировано одно событие:

- **Дата:** 24 апреля 2026 г.  
- **Время:** 10:15 (UTC‑04:00)  
- **Описание:** сдать лабу  

Больше событий в этом диапазоне нет.
